In [86]:
# --- Basic Imports ---
from utils import *
import warnings
warnings.filterwarnings("ignore", message=".*serialized model.*")
import shutil

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score,
                             precision_recall_fscore_support, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, log_loss)

from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from category_encoders import WOEEncoder
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier, Booster
import xgboost as xgb
import shap
import json


In [87]:
def find_best_threshold(y_true, y_prob, metric="f1"):
    thresholds = np.linspace(0.01, 0.99, 200)
    scores = []

    for t in thresholds:
        y_pred = (y_prob > t).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )

        if metric == "precision":
            scores.append(precision)
        elif metric == "recall":
            scores.append(recall)
        else:  # default f1
            scores.append(f1)

    scores = np.array(scores)
    best_idx = scores.argmax()
    best_threshold = thresholds[best_idx]
    best_score = scores[best_idx]

    return best_threshold, best_score, thresholds, scores, metric

In [88]:
def replace_unknowns(df):
    return df.replace("unknown", np.nan)

In [89]:
models = os.listdir(GOLD_PATH)
models = [f for f in models if f.endswith('.pkl')]

best_model_file = None
best_f1 = -1

for model_file in models:
    try:
        model_path = os.path.join(GOLD_PATH, model_file)
        # print(f"Loading model from: {model_path}")
        model = ModelWrapper.load(model_path)
        f1_score = model.metadata['F1']
        # print(f"Evaluating model: {model_file}, F1: {f1_score}")
        if f1_score >= best_f1:
            best_f1 = f1_score
            best_model_file = model_file
    except Exception as e:
        print(f"Failed to load/evaluate model {model_file}: {e}")

print(f"Best model: {best_model_file} with F1: {best_f1}")

Model loaded from ../01-data/03-gold/Logistic_Regression_Model_20251201_002836.pkl
Model loaded from ../01-data/03-gold/Logistic_Regression_Model_20251201_003532.pkl
Model loaded from ../01-data/03-gold/Logistic_Regression_Model_20251201_142022.pkl
Model loaded from ../01-data/03-gold/Logistic_Regression_Model_20251202_223734.pkl
Model loaded from ../01-data/03-gold/Logistic_Regression_Model_20251205_153411.pkl
Model loaded from ../01-data/03-gold/XGBoost_Model_20251201_181913.pkl
Model loaded from ../01-data/03-gold/XGBoost_Model_20251202_225036.pkl
Model loaded from ../01-data/03-gold/XGBoost_Model_20251203_202325.pkl
Model loaded from ../01-data/03-gold/XGBoost_Model_20251205_154421.pkl
Best model: XGBoost_Model_20251203_202325.pkl with F1: 0.5071982281284607


For consistency with Gradio we are going to handpick the best model from the previous query

In [90]:
# From best_model_file, extract date and time, convert to yyyy-MM-mm hh:mm:ss format
timestamp = best_model_file.split('_')[2] + '_' + best_model_file.split('_')[3].split('.')[0]
timestamp = dt.datetime.strptime(timestamp, "%Y%m%d_%H%M%S")
timestamp_str = timestamp.strftime("%Y-%m-%d %H:%M:%S")

In [91]:
best_model_file = 'XGBoost_Model_20251205_154421.pkl'

best_model_path = os.path.join(GOLD_PATH, 'XGBoost_Model_20251205_154421.pkl')
model = ModelWrapper.load(best_model_path)

# Get feature names from the best model, remove num__ prefix and drop features that end with __PCA0 or 1 suffix
feature_names = [feat.replace('num__', '') for feat in model.metadata['Feature Names'] if not (feat.endswith('__pca0') or feat.endswith('__pca1'))] + ['emp_var_rate','euribor3m','nr_employed','cons_price_idx']

# From best_model_file, extract date and time, convert to yyyy-MM-mm hh:mm:ss format
timestamp = best_model_file.split('_')[2] + '_' + best_model_file.split('_')[3].split('.')[0]
timestamp = dt.datetime.strptime(timestamp, "%Y%m%d_%H%M%S")
timestamp_str = timestamp.strftime("%Y-%m-%d %H:%M:%S")

# Load the XGBoost.csv file for the timestamp identified
df = pd.read_csv(os.path.join(MODEL_DIR, 'XGBoost_Model/XGBoost_metrics.csv'), sep=";")
df = df[df['Timestamp'] == timestamp_str]
hyper_params = df.iloc[0]['Best Parameters']

# Convert hyper_params string to dictionary
hyper_params = json.loads(hyper_params.replace("'", '"'))

# Remove from hyper_params any keys that contain select__k
hyper_params = {k: v for k, v in hyper_params.items() if 'select__k' not in k}


Model loaded from ../01-data/03-gold/XGBoost_Model_20251205_154421.pkl


In [102]:
print(sorted(feats))
print(sorted(app))

['age_emp_rate', 'cons_price_idx', 'contact', 'emp_var_rate', 'euribor3m', 'euribor_nrm', 'job', 'job_x_age', 'month_cos', 'nr_employed', 'pdays', 'poutcome', 'previous']
['age', 'age_emp_rate', 'cons_price_idx', 'contact', 'emp_var_rate', 'euribor3m', 'euribor_nrm', 'job', 'job_x_age', 'month_cos', 'nr_employed', 'pdays', 'poutcome', 'previous']


In [104]:
X.columns

Index(['pdays', 'previous', 'month_cos', 'age_emp_rate', 'euribor_nrm', 'job',
       'contact', 'poutcome', 'job_x_age', 'emp_var_rate', 'euribor3m',
       'nr_employed', 'cons_price_idx'],
      dtype='object')

In [93]:
df = pd.read_csv(os.path.join(SILVER_PATH, 'base_dataset_XGB_Engineered.csv'), sep=';')

X = df[feature_names].copy()
y = df['y'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

macro_cols = ['emp_var_rate','euribor3m','nr_employed','cons_price_idx']

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_features = [col for col in numeric_features if col not in macro_cols]
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Preprocessing pipelines for numeric and categorical data
pca_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("woe", WOEEncoder(handle_missing="value", handle_unknown="value"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("macro_economics", pca_pipeline, macro_cols),
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop"
)

cleaner = FunctionTransformer(replace_unknowns, validate=False, feature_names_out="one-to-one")

# SMOTE approach with hyper_params dictionary
clf = Pipeline(steps=[
    ("clean", cleaner),
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42,
                    k_neighbors=hyper_params["smote__k_neighbors"],
                    sampling_strategy=hyper_params["smote__sampling_strategy"])),
    ("model", XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        colsample_bytree=hyper_params["model__colsample_bytree"],
        learning_rate=hyper_params["model__learning_rate"],
        max_depth=hyper_params["model__max_depth"],
        min_child_weight=hyper_params["model__min_child_weight"],
        n_estimators=hyper_params["model__n_estimators"],
        subsample=hyper_params["model__subsample"],
        random_state=42
    ))
])

clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)[:, 1]

best_threshold, best_score, thresholds, scores, metric = find_best_threshold(
    y_test, y_prob, metric="f1"
)

# Apply best threshold
y_pred_opt = (y_prob > best_threshold).astype(int)

print("\nClassification Report with Best Threshold:")
print(classification_report(y_test, y_pred_opt))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_opt))

model = ModelWrapper(
    pipeline=clf,
    threshold=best_threshold,
)

model.save(HUGGING_DIR + "model.pkl")


Classification Report with Best Threshold:
              precision    recall  f1-score   support

           0       0.94      0.91      0.93      7265
           1       0.47      0.56      0.51       971

    accuracy                           0.87      8236
   macro avg       0.70      0.74      0.72      8236
weighted avg       0.88      0.87      0.88      8236

Confusion Matrix:
[[6638  627]
 [ 426  545]]
Model saved to ../96-huggingface_space/model.pkl
